In [18]:
# ── EXP-04: DeepLabV3 (ResNet-101 backbone) + Focal Loss + Augmentation ────
data_dir    = 'xBD_UC3M'
patch_size  = 512    # larger patches → more context per sample
num_epochs  = 25
batch_size  = 2      # 512×512 inputs need more memory
num_workers = 0

BASE_RESULT_DIR = r'/Users/juan.macias@feverup.com/Desktop/cv/cv-2a-image-segmentation/VA_Pr2A_ImageSegmentation_2025_2026/results/parte-4'
result_dir      = os.path.join(BASE_RESULT_DIR, 'exp04_deeplabv3_focal_aug')
model_name      = 'deeplabv3_focal_aug_exp04'

num_classes = 2
class_names = ['background', 'building']

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

# ── Dataset statistics (cached from EXP-01) ───────────────────────────────────
stats_cache = os.path.join(BASE_RESULT_DIR, 'xbd_stats.npy')
image_stats = np.load(stats_cache, allow_pickle=True).item()
print(f"Stats loaded — mean={image_stats['mean']}  std={image_stats['std']}")

# ── Datasets ──────────────────────────────────────────────────────────────────
# Note: larger patch_size + larger stride → fewer but richer samples per epoch.
train_ds_raw = xBDDataset(data_dir, split=['train'], task='segmentation',
                           patch_size=patch_size, stats=None, transform=None)
val_ds_raw   = xBDDataset(data_dir, split=['val'],   task='segmentation',
                           patch_size=patch_size, stats=None, transform=None)
test_ds_raw  = xBDDataset(data_dir, split=['test'],  task='segmentation',
                           patch_size=patch_size, stats=None, transform=None)


# ── KeyAdapter with augmentation (same logic as EXP-03) ──────────────────────
# ImageNet normalization constants (pretrained ResNet-101 expects this distribution)
_IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class KeyAdapterBinaryAug(Dataset):
    def __init__(self, ds, augment=False):
        self.ds      = ds
        self.augment = augment

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        s     = self.ds[idx]
        # stats=None → xBDDataset returns pixels in [0, 1] (raw/255)
        # Apply ImageNet normalization so pretrained BN running stats are valid
        image = (s['patch_post'].clone() - _IMAGENET_MEAN) / _IMAGENET_STD
        mask  = (s['mask_patch_raw'] != 255).long()

        if self.augment:
            if random.random() > 0.5:
                image = torch.flip(image, [2])
                mask  = torch.flip(mask,  [1])
            if random.random() > 0.5:
                image = torch.flip(image, [1])
                mask  = torch.flip(mask,  [0])
            k = random.randint(0, 3)
            if k:
                image = torch.rot90(image, k, [1, 2])
                mask  = torch.rot90(mask,  k, [0, 1])
            # Photometric jitter on ImageNet-normalized tensors
            image = image + random.uniform(-0.1, 0.1)
            for c in range(image.shape[0]):
                mu = image[c].mean()
                image[c] = (image[c] - mu) * random.uniform(0.8, 1.2) + mu

        return {'image': image, 'mask': mask, 'img_path': s['patch_post_path']}


dataloaders = {
    'Train': DataLoader(KeyAdapterBinaryAug(train_ds_raw, augment=True),  batch_size=batch_size,
                        shuffle=True,  num_workers=num_workers),
    'Val':   DataLoader(KeyAdapterBinaryAug(val_ds_raw,   augment=False), batch_size=1,
                        shuffle=False, num_workers=num_workers),
    'Test':  DataLoader(KeyAdapterBinaryAug(test_ds_raw,  augment=False), batch_size=1,
                        shuffle=False, num_workers=num_workers),
}


# ── Focal Loss (same as EXP-03) ───────────────────────────────────────────────
class FocalLoss(torch.nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, inputs, targets):
        ce    = torch.nn.functional.cross_entropy(inputs, targets,
                                                  weight=self.alpha, reduction='none')
        p_t   = torch.exp(-ce)
        focal = (1.0 - p_t) ** self.gamma * ce
        return focal.mean()


from torchvision.models.segmentation import deeplabv3_resnet101, DeepLabV3_ResNet101_Weights
from torchvision.models.segmentation.fcn import FCNHead

def get_deeplabv3_r101(num_classes):
    model = deeplabv3_resnet101(weights=DeepLabV3_ResNet101_Weights.DEFAULT)
    model.classifier     = DeepLabHead(2048, num_classes)
    model.aux_classifier = FCNHead(1024, num_classes)
    return model


Device: mps
Stats loaded — mean=[94.487564 97.861465 77.18053 ]  std=[39.98483  34.140186 35.44793 ]

[xBDDataset] split=['train']  task=segmentation  patch_size=512  samples=254
  Building annotations across all windows:
    no-damage      :   13216
    minor-damage   :    1117
    major-damage   :     617
    destroyed      :     864
    unlabelled     :       0


[xBDDataset] split=['val']  task=segmentation  patch_size=512  samples=45
  Building annotations across all windows:
    no-damage      :    1314
    minor-damage   :     228
    major-damage   :     242
    destroyed      :     143
    unlabelled     :       0


[xBDDataset] split=['test']  task=segmentation  patch_size=512  samples=63
  Building annotations across all windows:
    no-damage      :    4957
    minor-damage   :     516
    major-damage   :     153
    destroyed      :     324
    unlabelled     :       0



In [ ]:
model = get_deeplabv3_r101(num_classes).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# ── Focal Loss — alpha from measured pixel ratio (4.6x) ──────────────────────
alpha     = torch.tensor([1.0, 4.6]).to(device)
criterion = FocalLoss(gamma=2.0, alpha=alpha)

# ── Optimiser: lower LR for fine-tuning a pretrained backbone ─────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
lr_sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
metrics   = {'jaccard_score': jaccard_score}

os.makedirs(result_dir, exist_ok=True)

# ── Train ─────────────────────────────────────────────────────────────────────
model = train_model(
    model, criterion, dataloaders, device,
    optimizer, lr_sched, metrics,
    result_dir, model_name,
    num_classes=num_classes, num_epochs=num_epochs,
)

# ── Evaluate on test set ──────────────────────────────────────────────────────
weights = torch.load(os.path.join(result_dir, model_name + '_best.pth.tar'))['state_dict']
model.to(device)
model.load_state_dict(weights)
model.eval()
cm_total = test_segmentation_model(model, dataloaders, num_classes, class_names, result_dir, True, batchsize_test)


Parameters: 60,986,436
Epoch 1/25
----------


100%|██████████| 127/127 [01:10<00:00,  1.81it/s]


Train Loss: 0.3745


100%|██████████| 45/45 [00:05<00:00,  8.85it/s]


Val Loss: 0.4246
{'epoch': 1, 'Train_loss': 0.3745399827328254, 'Val_loss': 0.4245715700917774, 'Train_jaccard_score': np.float64(0.32335656337681257), 'Val_jaccard_score': np.float64(0.3358343156930137)}
Epoch 2/25
----------


...

100%|██████████| 45/45 [00:04<00:00,  9.03it/s]


Val Loss: 1.0392
{'epoch': 2, 'Train_loss': 0.27421564426947764, 'Val_loss': 1.0391862991783354, 'Train_jaccard_score': np.float64(0.4053519033117226), 'Val_jaccard_score': np.float64(0.2801527361650121)}
Epoch 3/25
----------


100%|██████████| 127/127 [01:09<00:00,  1.83it/s]


Train Loss: 0.2647


 76%|███████▌  | 34/45 [00:03<00:01,  8.95it/s]